## 1) Objective

Professional data quality validation for Instacart raw data.

This notebook provides a repeatable quality workflow:
- standardized table profiling
- missing and duplicate diagnostics
- key integrity checks (PK/FK)
- domain/range rule validation
- issue register with severity and status
- exportable quality reports

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../../data/raw/')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

In [11]:
orders = pd.read_csv(DATA_PATH / 'orders.csv')
order_products_prior = pd.read_csv(DATA_PATH / 'order_products__prior.csv')
order_products_train = pd.read_csv(DATA_PATH / 'order_products__train.csv')
products = pd.read_csv(DATA_PATH / 'products.csv')
aisles = pd.read_csv(DATA_PATH / 'aisles.csv')
departments = pd.read_csv(DATA_PATH / 'departments.csv')

tables = {
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products,
    'aisles': aisles,
    'departments': departments,
}

print(f'Loaded {len(tables)} tables from: {DATA_PATH.resolve()}')

Loaded 6 tables from: /home/ppt/Desktop/DataAnalyst/Instacart-Market-Basket-Analysis/data/raw


## 2) Quality Check Framework

In [12]:
def profile_tables(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in table_dict.items():
        rows.append(
            {
                'table': name,
                'rows': df.shape[0],
                'cols': df.shape[1],
                'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
                'duplicate_rows': int(df.duplicated().sum()),
                'missing_cells': int(df.isna().sum().sum()),
                'missing_pct_cells': round((df.isna().sum().sum() / df.size) * 100, 4),
            }
        )
    return pd.DataFrame(rows).sort_values(['missing_cells', 'duplicate_rows'], ascending=False)


def column_missing(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    records = []
    for name, df in table_dict.items():
        for col in df.columns:
            miss_count = int(df[col].isna().sum())
            if miss_count > 0:
                records.append(
                    {
                        'table': name,
                        'column': col,
                        'missing_count': miss_count,
                        'missing_pct': round(df[col].isna().mean() * 100, 4),
                    }
                )
    if not records:
        return pd.DataFrame(columns=['table', 'column', 'missing_count', 'missing_pct'])
    return pd.DataFrame(records).sort_values(['missing_pct', 'missing_count'], ascending=False)


def check_pk_unique(df: pd.DataFrame, table_name: str, key_col: str) -> dict:
    non_null_count = int(df[key_col].notna().sum())
    unique_non_null = int(df[key_col].nunique(dropna=True))
    pass_flag = bool(df[key_col].is_unique and non_null_count == len(df))
    return {
        'check_type': 'PK_UNIQUENESS',
        'table': table_name,
        'column': key_col,
        'status': 'PASS' if pass_flag else 'FAIL',
        'invalid_count': int(len(df) - unique_non_null),
        'severity': 'critical',
        'detail': f'non_null={non_null_count}, unique_non_null={unique_non_null}, rows={len(df)}',
    }


def check_fk_coverage(child_df: pd.DataFrame, child_table: str, child_col: str, parent_df: pd.DataFrame, parent_table: str, parent_col: str, severity: str = 'critical') -> dict:
    coverage = float(child_df[child_col].isin(parent_df[parent_col]).mean() * 100)
    invalid_count = int((~child_df[child_col].isin(parent_df[parent_col])).sum())
    pass_flag = invalid_count == 0
    return {
        'check_type': 'FK_COVERAGE',
        'table': child_table,
        'column': child_col,
        'status': 'PASS' if pass_flag else 'FAIL',
        'invalid_count': invalid_count,
        'severity': severity,
        'detail': f'{child_table}.{child_col} in {parent_table}.{parent_col} = {coverage:.4f}%',
    }


def check_range(df: pd.DataFrame, table_name: str, col: str, min_val: float, max_val: float, severity: str = 'major') -> dict:
    invalid_mask = ~df[col].between(min_val, max_val)
    invalid_count = int(invalid_mask.sum())
    return {
        'check_type': 'RANGE',
        'table': table_name,
        'column': col,
        'status': 'PASS' if invalid_count == 0 else 'FAIL',
        'invalid_count': invalid_count,
        'severity': severity,
        'detail': f'expected between [{min_val}, {max_val}]',
    }


def check_allowed_set(df: pd.DataFrame, table_name: str, col: str, allowed_values: set, severity: str = 'major') -> dict:
    invalid_mask = ~df[col].isin(allowed_values)
    invalid_count = int(invalid_mask.sum())
    return {
        'check_type': 'ALLOWED_SET',
        'table': table_name,
        'column': col,
        'status': 'PASS' if invalid_count == 0 else 'FAIL',
        'invalid_count': invalid_count,
        'severity': severity,
        'detail': f'allowed={sorted(allowed_values)}',
    }

## 3) Profiling and Missing Diagnostics

In [13]:
missing_df = column_missing(tables)
missing_df.head(30)

,table,column,missing_count,missing_pct
0,orders,days_since_prior_order,206209,6.0276


In [14]:
profile_df = profile_tables(tables)
profile_df

,table,rows,cols,memory_mb,duplicate_rows,missing_cells,missing_pct_cells
0,orders,3421083,7,332.7100,0,206209,0.8611
1,order_products_prior,32434489,4,989.8200,0,0,0.0000
2,order_products_train,1384617,4,42.2600,0,0,0.0000
3,products,49688,4,4.9300,0,0,0.0000
4,aisles,134,2,0.0100,0,0,0.0000
5,departments,21,2,0.0000,0,0,0.0000


## 4) Integrity 

In [15]:
# เช็คจำนวน User ทั้งหมดที่มีในระบบ
total_users = tables['orders']['user_id'].nunique()

# เช็คจำนวนแถวที่เป็น Missing (ครั้งแรกของการสั่งซื้อ)
first_orders = (tables['orders']['order_number'] == 1).sum()

print(f"จำนวน User ทั้งหมด: {total_users:,}")
print(f"จำนวน Order ครั้งแรก: {first_orders:,}")

จำนวน User ทั้งหมด: 206,209
จำนวน Order ครั้งแรก: 206,209


In [16]:
clean_tables = {name: df.copy() for name, df in tables.items()}
imputation_log = []


def apply_fill(df: pd.DataFrame, table: str, column: str, fill_value, reason: str, cast_dtype: str | None = None):
    before_missing = int(df[column].isna().sum())
    df[column] = df[column].fillna(fill_value)
    if cast_dtype is not None:
        df[column] = df[column].astype(cast_dtype)
    after_missing = int(df[column].isna().sum())
    imputation_log.append({
        'table': table,
        'column': column,
        'fill_value': str(fill_value),
        'missing_before': before_missing,
        'missing_after': after_missing,
        'filled_count': before_missing - after_missing,
        'reason': reason,
    })


# Business-safe imputation for Instacart: first order has no prior-gap by definition
apply_fill(
    clean_tables['orders'],
    table='orders',
    column='days_since_prior_order',
    fill_value=0,
    reason='First order has no previous order; set gap to 0 day for modeling compatibility.',
    cast_dtype='float32',
)

# Defensive string imputations (applied only when nulls exist)
if int(clean_tables['products']['product_name'].isna().sum()) > 0:
    apply_fill(
        clean_tables['products'],
        table='products',
        column='product_name',
        fill_value='Unknown Product',
        reason='Preserve product row when name is missing.',
    )

if int(clean_tables['aisles']['aisle'].isna().sum()) > 0:
    apply_fill(
        clean_tables['aisles'],
        table='aisles',
        column='aisle',
        fill_value='Unknown Aisle',
        reason='Preserve dimension row when aisle name is missing.',
    )

if int(clean_tables['departments']['department'].isna().sum()) > 0:
    apply_fill(
        clean_tables['departments'],
        table='departments',
        column='department',
        fill_value='Unknown Department',
        reason='Preserve dimension row when department name is missing.',
    )

imputation_df = pd.DataFrame(imputation_log)
imputation_df

,table,column,fill_value,missing_before,missing_after,filled_count,reason
0,orders,days_since_prior_order,0,206209,0,206209,First order has no previous order; set gap to ...


## 6) Check Duplicate and Missing Value

In [17]:
# 1. สร้างสรุปของข้อมูลก่อนทำความสะอาด (Raw/Old)
raw_summary = []
for name, df in tables.items():
    raw_summary.append({
        'table': name,
        'missing_before': df.isna().sum().sum(),
        'dup_before': df.duplicated().sum()
    })
raw_df = pd.DataFrame(raw_summary)

# 2. สร้างสรุปของข้อมูลหลังทำความสะอาด (Cleaned)
clean_summary = []
for name, df in clean_tables.items():
    clean_summary.append({
        'table': name,
        'missing_after': df.isna().sum().sum(),
        'dup_after': df.duplicated().sum()
    })
clean_df = pd.DataFrame(clean_summary)

# 3. Merge เข้าด้วยกันแล้วคำนวณส่วนต่าง
comparison = pd.merge(raw_df, clean_df, on='table')
comparison['filled_missing'] = comparison['missing_before'] - comparison['missing_after']
comparison['removed_duplicates'] = comparison['dup_before'] - comparison['dup_after']

# แสดงผลเฉพาะคอลัมน์ที่สำคัญ
display(comparison[['table', 'filled_missing', 'removed_duplicates']])

,table,filled_missing,removed_duplicates
0,orders,206209,0
1,order_products_prior,0,0
2,order_products_train,0,0
3,products,0,0
4,aisles,0,0
5,departments,0,0


## Save Clean Data

In [18]:
orders = pd.read_csv(DATA_PATH / 'orders.csv')
order_products_prior = pd.read_csv(DATA_PATH / 'order_products__prior.csv')
order_products_train = pd.read_csv(DATA_PATH / 'order_products__train.csv')
products = pd.read_csv(DATA_PATH / 'products.csv')
aisles = pd.read_csv(DATA_PATH / 'aisles.csv')
departments = pd.read_csv(DATA_PATH / 'departments.csv')

tables = {
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products,
    'aisles': aisles,
    'departments': departments,
}

print(f'Loaded {len(tables)} tables from: {DATA_PATH.resolve()}')

Loaded 6 tables from: /home/ppt/Desktop/DataAnalyst/Instacart-Market-Basket-Analysis/data/raw


In [19]:
processed_path = "../../data/processed/quality/"

# เซฟเฉพาะตารางหลักที่คุณต้องใช้ (เน้นที่แก้บ่อยๆ)
clean_tables['orders'].to_csv(f"{processed_path}orders_clean.csv", index=False)
clean_tables['order_products_prior'].to_csv(f"{processed_path}order_products_prior_clean.csv", index=False)
clean_tables['order_products_train'].to_csv(f"{processed_path}order_products_train_clean.csv", index=False)
clean_tables['products'].to_csv(f"{processed_path}products_clean.csv", index=False)
clean_tables['aisles'].to_csv(f"{processed_path}aisles_clean.csv", index=False)
clean_tables['departments'].to_csv(f"{processed_path}departments_clean.csv", index=False)

# สำหรับตารางอื่นๆ ถ้าคุณไม่ได้แก้ไขอะไรเลย (เช่น aisles, departments) 
# จะเซฟใหม่ หรือจะใช้ไฟล์เดิมจาก data/raw ก็ได้ครับ ไม่ผิด